# Import packages and data loading

In [1]:
import time
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src import (
    load_instance,
    build_model,
    solve_model,
    solve_and_summarize_all,
    extract_assignment,
    print_summary,
    get_model_stats,
    display_solution,
)

import json
import pandas as pd

in this section we work with models that use Hungarian policy to solve problems with possible ties. bacause of this, we don't need preprocessed datasets and the simple datasets are enough.

In [2]:
data_small = json.load(open(r'../data/generated/instance_small.json', 'r', encoding='utf-8'))
print("Loaded small instance:\n   n={}, m={}".format(data_small['n'], data_small['m']))

data_medium = json.load(open(r'../data/generated/instance_medium.json', 'r', encoding='utf-8'))
print("Loaded medium instance:\n   n={}, m={}".format(data_medium['n'], data_medium['m']))

Loaded small instance:
   n=10, m=5
Loaded medium instance:
   n=50, m=20


# Solving Models

## Using Simple Datasets
we are going to use Pyomo models that implemented the formulations below from the paper:

- `MIN-CUT`
- `MSMR-CUT`
- `MIN-BIN-CUT`
- `MSMR-BIN-CUT`
- `SO-H-NW-CUT`
- `SO-H-NW-BIN-CUT`

In [3]:
section_three_formulations = ["MIN-CUT", "MSMR-CUT", "MIN-BIN-CUT", "MSMR-BIN-CUT", "SO-H-NW-CUT", "SO-H-NW-BIN-CUT"]

results_small = solve_and_summarize_all(data_small, solver_name='cplex', tee=False, formulations=section_three_formulations)


print("\nSUMMARY TABLE")
print("="*100)

df_summary = pd.DataFrame([
    {
        'Formulation': r['model'],
        'Status': r['status'].upper(),
        'Model Obj': f"{r['model_objective']:.2f}" if r['model_objective'] is not None else "N/A",
        'Rank Obj': f"{r['rank_objective']}" if r['rank_objective'] is not None else "N/A",
        'Matched': r['students_assigned'],
        'Avg Rank': f"{r['rank_objective']/max(1, r['students_assigned']):.2f}" if r['rank_objective'] is not None and r['students_assigned'] > 0 else "N/A"
    }
    for r in results_small
])

print(df_summary.to_string(index=False))


SUMMARY TABLE
    Formulation  Status Model Obj Rank Obj  Matched Avg Rank
        MIN-CUT OPTIMAL     79.00        8        4     2.00
       MSMR-CUT OPTIMAL     12.00        8        4     2.00
    MIN-BIN-CUT OPTIMAL      5.00        8        4     2.00
   MSMR-BIN-CUT OPTIMAL     12.00        8        4     2.00
    SO-H-NW-CUT OPTIMAL     19.00        6        5     1.20
SO-H-NW-BIN-CUT OPTIMAL     12.00        8        4     2.00


In [4]:
solution_details = {}

for formulation in section_three_formulations:
    info = solve_model(data_medium, formulation=formulation, solver_name='cplex', tee=False)
    model = info['model']
    status = info['status']
    
    if 'optimal' in status or 'feasible' in status:
        metrics = display_solution(formulation, model, data_medium)
        solution_details[formulation] = metrics
    else:
        print(f"\n{formulation}: INFEASIBLE or DID NOT SOLVE")


Formulation: MIN-CUT
Objective value: 255.00
Students matched: 50/50
Unmatched students: 0
Average preference rank: 0.68
Max preference rank: 5
Total rank objective: 34
College loads: {0: 3, 1: 2, 2: 4, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 4, 11: 4, 12: 3, 13: 2, 14: 2, 15: 3, 16: 1, 17: 1, 18: 1, 19: 2}
Assignments: {0: 3, 1: 2, 2: 2, 3: 11, 4: 10, 5: 10, 6: 11, 7: 17, 8: 15, 9: 19, 10: 14, 11: 18, 12: 12, 13: 0, 14: 3, 15: 7, 16: 14, 17: 10, 18: 8, 19: 10, 20: 11, 21: 9, 22: 0, 23: 13, 24: 12, 25: 4, 26: 3, 27: 2, 28: 6, 29: 5, 30: 15, 31: 6, 32: 2, 33: 7, 34: 1, 35: 0, 36: 13, 37: 9, 38: 3, 39: 5, 40: 19, 41: 8, 42: 4, 43: 11, 44: 15, 45: 1, 46: 4, 47: 9, 48: 12, 49: 16}

Formulation: MSMR-CUT
Objective value: 966.00
Students matched: 50/50
Unmatched students: 0
Average preference rank: 0.68
Max preference rank: 5
Total rank objective: 34
College loads: {0: 3, 1: 2, 2: 4, 3: 4, 4: 3, 5: 2, 6: 2, 7: 2, 8: 2, 9: 3, 10: 4, 11: 4, 12: 3, 13: 2, 14: 2, 15: 3, 16: 1, 17: 1, 18: 

# Computational performance and comparison

in the cell below, we solve the 'medium dataset' with all the models, and using the function `get_model_stats()`, we can observe four different items:

- total number of variables for each model
- total number of constraints for each model
- size of the model's LP file in kB
- duration time for each model to be solved

the original paper used the same indices in the table 3, along with "non-0 elem" which shows the number of non-zero parameters for each model. because of the problems with finding these values from a model solved with `clpex`, I gave up on these stats.

In [5]:
rows = []

for formulation in section_three_formulations:
    print(f"Solving {formulation} on strict medium...")
    start = time.perf_counter()
    info = solve_model(data_medium, formulation=formulation, solver_name='cplex', tee=False)
    elapsed = time.perf_counter() - start
    model = info['model']
    stats = get_model_stats(model)
    rows.append({
        'Formulation': formulation,
        '#variables': stats['num_vars'],
        '#constraints': stats['num_constraints'],
        'size(Kb)': f"{stats['size_kb']:.2f}",
        'run time(s)': f"{elapsed:.2f}"
    })

# create data frame and print
df_table = pd.DataFrame(rows)
print("\nTable 3: The performances of (mixed) integer programming formulations for the case of ties.")
print("="*100)
print(df_table.to_string(index=False))

Solving MIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-CUT on strict medium...
not match specified file format (lp)
Solving MIN-BIN-CUT on strict medium...
not match specified file format (lp)
Solving MSMR-BIN-CUT on strict medium...
not match specified file format (lp)
Solving SO-H-NW-CUT on strict medium...
not match specified file format (lp)
Solving SO-H-NW-BIN-CUT on strict medium...
not match specified file format (lp)

Table 3: The performances of (mixed) integer programming formulations for the case of ties.
    Formulation  #variables  #constraints size(Kb) run time(s)
        MIN-CUT        1020          2070   199.34        0.70
       MSMR-CUT        1020          2070   210.76        0.60
    MIN-BIN-CUT        1647          2697   232.83        0.64
   MSMR-BIN-CUT        1647          2697   229.61        0.68
    SO-H-NW-CUT        2040         13610   661.14        1.49
SO-H-NW-BIN-CUT        2647         14188   686.69        1.46


In [6]:
data_large = json.load(open(r'../data/generated/instance_large.json', 'r', encoding='utf-8'))
print("Loaded strictly-ranked large instance:\n   n={}, m={}".format(data_large['n'], data_large['m']))

Loaded strictly-ranked large instance:
   n=1000, m=20


In [ ]:
rows = []

for formulation in section_three_formulations:
    print(f"Solving {formulation} on strict large...")
    start = time.perf_counter()
    info = solve_model(data_large, formulation=formulation, solver_name='cplex', tee=False)
    elapsed = time.perf_counter() - start
    model = info['model']
    stats = get_model_stats(model)
    rows.append({
        'Formulation': formulation,
        '#variables': stats['num_vars'],
        '#constraints': stats['num_constraints'],
        'size(Kb)': f"{stats['size_kb']:.2f}",
        'run time(s)': f"{elapsed:.2f}"
    })

# create data frame and print
df_table = pd.DataFrame(rows)
print("\nTable 3: The performances of (mixed) integer programming formulations for the case of ties.")
print("="*100)
print(df_table.to_string(index=False))

Solving MIN-CUT on strict large...
